In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Controls — immediately after Drive mount. Safe for a brand-new Runtime → Run all.
DRIVE_ROOT='/content/drive/MyDrive/OpenPlaque'
DICOM_ROOT='/content/drive/MyDrive/CCTA/DICOM'
OUTPUT_ROOT=DRIVE_ROOT + '/LCX_Curved_Template_Reacquisition_v1_fixed'
MINI_ZIP=DRIVE_ROOT + '/Cache/LCX_Curved_Template_DICOM_v1/curved_series_1035_1039.zip'
BRANCH='lcx-curved-template-reacquisition-from-main'
TARGET_SERIES=(1035, 1039)


# OpenPlaque — LCX curved-template reacquisition (fixed DICOM-source notebook)

This is the same LCX curved-template experiment, with the runtime input bug fixed. The previous notebook assumed `OpenPlaque/Full_DICOM.zip`; that archive is not present. This notebook instead finds Series **1035 (RCA)** and **1039 (historical LCX-labeled curved series)** in the live `MyDrive/CCTA/DICOM` tree, builds/reuses a small persistent two-series cache ZIP, and passes that archive to the unchanged scientific workflow.

The historical LCX mask remains an **unlabeled coronary template**. A template match does not establish LCX identity or source-space plaque volume.


In [ ]:
import shutil, subprocess, sys
shutil.rmtree('/content/OpenPlaque', ignore_errors=True)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/pazzani/OpenPlaque.git','/content/OpenPlaque'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','pytest','scikit-image','pydicom','SimpleITK'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','/content/OpenPlaque'], check=True)
# Separate-process package import verification.
subprocess.run([sys.executable,'-c','import openplaque; import openplaque.lcx_curved_template_reacquisition as m; print(openplaque.__file__); print(m.ALGORITHM)'], check=True)
subprocess.run(['git','-C','/content/OpenPlaque','rev-parse','HEAD'], check=True)


In [ ]:
# Source integrity + experiment tests.
subprocess.run([sys.executable,'-m','py_compile','/content/OpenPlaque/src/openplaque/lcx_curved_template_reacquisition.py'], check=True)
subprocess.run([sys.executable,'-m','pytest','-q','/content/OpenPlaque/tests/test_lcx_curved_template_reacquisition.py'], check=True)


In [ ]:
# Preflight all non-DICOM inputs before touching the large DICOM tree.
from pathlib import Path
required = [
    Path(DRIVE_ROOT)/'Master_Coronary_Anatomy_Baseline_v2/master_anatomy_summary.json',
    Path(DRIVE_ROOT)/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.npy',
    Path(DRIVE_ROOT)/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.json',
    Path(DRIVE_ROOT)/'Cache/LAD_Frozen_Proximal_Reacquisition_v1/combined_lad_centerline.csv',
    Path(DRIVE_ROOT)/'PCAT_RCA_10_50/rca_centerline_smoothed_zyx.csv',
    Path(DRIVE_ROOT)/'TotalSegmentator_Cardiovascular_Cache_v1/coronary_arteries/coronary_arteries.nii.gz',
    Path(DRIVE_ROOT)/'TotalSegmentator_Cardiovascular_Cache_v1/coronary_arteries_LEGACY/coronary_arteries.nii.gz',
    Path(DRIVE_ROOT)/'TotalSegmentator_Cardiovascular_Cache_v1/total/aorta.nii.gz',
    Path(DRIVE_ROOT)/'UCLA_Plaque_Context_Verification/nnunet_masks/RCA.nii.gz',
    Path(DRIVE_ROOT)/'UCLA_Plaque_Context_Verification/nnunet_masks/LCX.nii.gz',
]
missing = [str(p) for p in required if not p.exists()]
print('Required inputs found:', len(required)-len(missing), '/', len(required))
if missing:
    raise FileNotFoundError('Missing required inputs:\n' + '\n'.join(missing))
if not Path(DICOM_ROOT).exists() and not Path(MINI_ZIP).exists():
    raise FileNotFoundError(f'Neither DICOM tree nor cached target-series ZIP exists: {DICOM_ROOT} ; {MINI_ZIP}')
print('DICOM tree:', DICOM_ROOT, 'exists=', Path(DICOM_ROOT).exists())
print('Target-series cache:', MINI_ZIP, 'exists=', Path(MINI_ZIP).exists())


In [ ]:
# Build a small persistent DICOM cache containing only Series 1035 and 1039.
# This avoids recreating or copying the full historical Full_DICOM.zip.
from pathlib import Path
import os, zipfile
import pydicom

mini = Path(MINI_ZIP)
mini.parent.mkdir(parents=True, exist_ok=True)

if mini.exists() and mini.stat().st_size > 0:
    print('Reusing target-series DICOM cache:', mini, mini.stat().st_size, 'bytes')
else:
    root = Path(DICOM_ROOT)
    wanted = set(int(x) for x in TARGET_SERIES)
    found = {n: [] for n in wanted}

    for dirpath, _, names in os.walk(root):
        if all(found[n] for n in wanted):
            break
        names = [n for n in names if not n.startswith('.')]
        if not names:
            continue

        # UCLA export stores a series together in a directory. Probe a few files;
        # if a target series is found, verify every file in that directory.
        probe_series = None
        for name in names[:5]:
            p = Path(dirpath) / name
            try:
                ds = pydicom.dcmread(str(p), stop_before_pixels=True, force=True, specific_tags=['SeriesNumber'])
                sn = int(getattr(ds, 'SeriesNumber', -1))
            except Exception:
                continue
            if sn in wanted:
                probe_series = sn
                break
        if probe_series is None or found[probe_series]:
            continue

        matches = []
        for name in names:
            p = Path(dirpath) / name
            try:
                ds = pydicom.dcmread(str(p), stop_before_pixels=True, force=True, specific_tags=['SeriesNumber'])
                if int(getattr(ds, 'SeriesNumber', -1)) == probe_series:
                    matches.append(p)
            except Exception:
                pass
        if matches:
            found[probe_series] = sorted(matches)
            print(f'Found Series {probe_series}: {len(matches)} files in {dirpath}')

    counts = {n: len(found[n]) for n in sorted(found)}
    print('Target-series file counts:', counts)
    missing_series = [n for n,c in counts.items() if c < 2]
    if missing_series:
        raise RuntimeError(f'Could not find complete target DICOM series: {missing_series}; counts={counts}')

    tmp = Path('/content/curved_series_1035_1039.tmp.zip')
    if tmp.exists():
        tmp.unlink()
    with zipfile.ZipFile(tmp, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=1) as zf:
        for sn in sorted(found):
            for i, p in enumerate(found[sn]):
                zf.write(p, arcname=f'series_{sn}/{i:04d}_{p.name}')
    shutil.copy2(tmp, mini)
    print('Created target-series DICOM cache:', mini, mini.stat().st_size, 'bytes')


In [ ]:
# Run the scientific workflow in a fresh Python process.
# Override only STUDY_ZIP with the validated two-series cache; scientific code is unchanged.
from pathlib import Path
import subprocess, sys, json
runner_path = Path('/content/run_lcx_curved_template_fixed.py')
runner_path.write_text(f'''\nfrom pathlib import Path\nimport shutil\nimport openplaque.lcx_curved_template_reacquisition as m\n\nshutil.rmtree('/content/openplaque_lcx_template_dicom', ignore_errors=True)\nm.STUDY_ZIP = Path({MINI_ZIP!r})\nprint('SELF TEST:', m.synthetic_lcx_template_self_test())\nprint('DICOM ARCHIVE:', m.STUDY_ZIP)\nresult = m.run({DRIVE_ROOT!r}, {OUTPUT_ROOT!r})\nprint('STATUS:', result['summary']['status'])\nprint('REPORT:', result['report'])\nprint('ZIP:', result['zip'])\nprint('Drive search: https://drive.google.com/drive/u/0/search?q=OPENPLAQUE_LCX_CURVED_TEMPLATE_REACQUISITION_REPORT_BACK.zip')\n''', encoding='utf-8')
proc = subprocess.run([sys.executable, str(runner_path)])
if proc.returncode != 0:
    state = Path(OUTPUT_ROOT)/'run_state.json'
    if state.exists():
        print('run_state.json:', state.read_text())
    raise RuntimeError(f'LCX workflow exited with code {proc.returncode}; the detailed Python traceback is immediately above.')


In [ ]:
# Display outputs after a successful run.
from pathlib import Path
from IPython.display import display, Image, HTML
out = Path(OUTPUT_ROOT)
for name in ['01_curved_template_fingerprints.png','02_candidate_match_scores.png','03_top_candidate_source_orthogonal_qc.png']:
    p = out/name
    if p.exists():
        display(Image(filename=str(p)))
report = out/'OPENPLAQUE_LCX_CURVED_TEMPLATE_REACQUISITION_REPORT.html'
zip_path = out/'OPENPLAQUE_LCX_CURVED_TEMPLATE_REACQUISITION_REPORT_BACK.zip'
print('REPORT:', report)
print('ZIP:', zip_path)
print('Drive search: https://drive.google.com/drive/u/0/search?q=OPENPLAQUE_LCX_CURVED_TEMPLATE_REACQUISITION_REPORT_BACK.zip')
